# Singapore MRT Ridership — Exploratory Data Analysis

This notebook explores the Singapore MRT passenger volume dataset from LTA DataMall.

# Objectives
- Understand the structure and shape of the data
- Identify trends, seasonality, and anomalies
- Prepare data for time-series modelling with ARIMA and Prophet

In [1]:
# ================================
# 1. Imports
# ================================
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Display plots inline in the notebook
%matplotlib inline

# Set default plot style
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (14, 5)

In [2]:
# ================================
# 2. Load Data
# ================================
import sys
sys.path.append("src")

from data_pipeline import load_raw_data
from utils import generate_date_range

In [3]:
# Generate list of all dates we expect to have
dates = generate_date_range("202301", "202412")
filenames = [f"{date}.csv" for date in dates]

# Load all raw data into one DataFrame
df = load_raw_data(filenames)

# Preview the data
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data\\raw\\202301.csv'

## 3. Data Overview

Understanding the structure, shape, and basic statistics of the dataset.

In [4]:
# ================================
# 3. Data Overview
# ================================

# Shape of the dataset
print("Shape:", df.shape)

# Column names
print("\nColumns:", df.columns.tolist())

# Data types
print("\nData Types:")
print(df.dtypes)

# Missing values
print("\nMissing Values:")
print(df.isnull().sum())

NameError: name 'df' is not defined

## 4. Time Series Visualization

Plotting ridership trends over time to identify patterns, seasonality, and anomalies such as the COVID-19 period.

In [5]:
# ================================
# 4. Time Series Visualisation
# ================================

# Aggregate total ridership per month
df["YEAR_MONTH"] = pd.to_datetime(df["YEAR_MONTH"], format="%Y%m")
monthly = df.groupby("YEAR_MONTH")["TOTAL_TAP_IN_VOLUME"].sum().reset_index()

# Plot
plt.figure(figsize=(14, 5))
plt.plot(monthly["YEAR_MONTH"], monthly["TOTAL_TAP_IN_VOLUME"], marker="o")
plt.title("Singapore MRT Monthly Ridership (2023-2024")
plt.xlabel("Month")
plt.ylabel("Total Tap In Volume")
plt.tight_layout()
plt.show()

NameError: name 'df' is not defined

## 5. Seasonality Analysis

Breaking down ridership patterns by day type (weekday vs weekend) and identifying recurring seasonal patterns.

In [6]:
# ================================
# 5. Seasonality Analysis
# ================================

# Separate weekday and weekend ridership
weekday = df[df["DAY_TYPE"] == "WEEKDAY"].groupby("YEAR_MONTH")["TOAL_TAP_IN_VOLUME"].sum().reset_index()
weekend = df[df["DAY_TYPE"] == "WEEKEND/PUBLIC HOLIDAY"].groupby("YEAR_MONTH")["TOTAL_TAP_IN_VOLUME"].sum().reset_index()

# Plot
plt.figure(figsize=(14, 5))
plt.plot(weekday["YEAR_MONTH"], weekday["TOTAL_TAP_IN_VOLUME"], marker="o", label="Weekday")
plt.plot(weekend["YEAR_MONTH"], weekend["TOTAL_TAP_IN_VOLUME"], marker="o", label="Weekend/PH")
plt.title("MRT Ridership — Weekday vs Weekend (2023-2024")
plt.xlabel("Month")
plt.ylabel("Total Tap In Volume")
plt.legend()
plt.tight_layout()
plt.show()

NameError: name 'df' is not defined

## 6. Top Stations by Ridership

Identifying the busiest MRT stations to understand which stations drive the most volume.

In [7]:
# ================================
# 6. Top Stations by Ridership
# ================================

# Aggregate total ridership per station
top_stations = df.groupby("PT_CODE")["TOTAL_TAP_IN_VOLUME"].sum().reset_index()
top_stations = top_stations.sort_values("TOTAL_TAP_IN_VOLUME", ascending=False).head(10)

# Plot
plt.figure(figsize=(14, 5))
sns.barplot(data=top_stations, x="PT_CODE", y="TOTAL_TAP_IN_VOLUME")
plt.title("Top 10 Busisest MRT Stations (2023-2024)")
plt.xlabel("Station Code")
plt.ylabel("Total Tap In Volume")
plt.tight_layout()
plt.show()

NameError: name 'df' is not defined

## 7. Anomaly Detection

Identifying unusual spikes or dips in ridership that may affect model performance and require special handling.

In [10]:
# ================================
# 7. Anomaly Detection
# ================================

# Calculate rolling mean and standard deviation
monthly["rolling_mean"] = monthly["TOTAL_TAP_IN_VOLUME"].rolling(windows=3).mean()
monthly["rolling_std"] = monthly["TOTAL_TAP_IN_VOLUME"].rolling(windows=3).std()

# Define upper and lower bounds (2 standard deviations)
monthly["upper_bound"] = monthly["rolling_mean"] + (2 * monthly["rolling_std"])
monthly["lower_bound"] = monthly["rolling_mean"] = (2 * monthly["rolling_std"])

# Plot
plt.figure(figsize=(14, 5))
plt.plot(monthly["YEAR_MONTH"], monthly["TOTAL_TAP_IN_VOLUME"], marker="o", label="Actual")
plt.plot(monthly["YEAR_MONTH"], monthly["rolling_mean"], linestyle="--", label="Rolling Mean")
plt.fill_between(monthly["YEAR_MONTH"], monthly["lower_bound"], monthly["upper_bound"], alpha=0.2, label="Normal Range")
plt.title("MRT Ridership — Anomaly Detection")
plt.xlabel("Month")
plt.ylabel("Total Tap In Volume")
plt.legend()
plt.tight_layout()
plt.show()

NameError: name 'monthly' is not defined

## 8. Data Export

Saving the cleaned and processed data to the 'data/processed/' folder, ready for modelling.

In [9]:
# ================================
# 8. Data Export
# ================================

# Convert YEAR_MONTH back to string for clean CSV export
monthly["YEAR_MONTH"] = monthly["YEAR_MONTH"].dt.strftime("%Y%m")

# Save processed data
os.makedirs("data/processed", exist_ok=True)
processed_filepath = os.path.join("data", "processed", "mrt_ridership_processed.csv")
monthly.to_csv(processed_filepath, index=False)

print(f"Processed data saved to {processed_filepath}")

NameError: name 'monthly' is not defined